# 5단계 추가: 공식 평가셋으로 평가 (논문에 없는 추가)
모델 카드에 링크된 `android_benchmark.xlsx` (논문 표 1 의 평가셋, 210개) 로 우리 모델과 공개 모델을 평가한다.
변환·점검 코드는 `src/official_bench.py`, 생성·채점은 5단계와 같은 `src/evaluate.py`. 이 노트북은 **실행만** 한다.

- 런타임: **A100 GPU**. 🔑 Secrets에 `HF_TOKEN` 등록
- 210개 = 함수 20개 × 10 + `irrelevant_function` 10 (negative 4.8%). 질의에 정답 형식이 대부분 들어 있다 (`17:15`, `+913322667890`, `'Nap Time'`)
- 결과는 Drive 의 `MyDrive/octopus_v2/runs/eval_official/` 에 저장 (변환한 `official_test.jsonl` 도 같이 복사 → 랩탑에서 다시 채점 가능)
- ⚠️ 논문 숫자(99.524% 등)는 nexa 가 사람 눈으로 T/F 를 매긴 관대한 기준이다. 같은 출력을 우리 채점기로 매기면
  Octopus-0 은 92.857% (6-1 에서 확인). 그래서 비교는 **같은 채점기로 매긴 숫자끼리** 한다

## 준비 (그냥 실행)

In [ ]:
# 5단계 노트북과 같은 버전 (최신 5.x 는 API 가 바뀌어 에러가 난다)
# 처음 설치한 뒤에는 '런타임 > 세션 다시 시작' 후 이 칸부터 다시 실행
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print("transformers", transformers.__version__)  # 4.56.2 여야 함

RUN_DIR = "/content/drive/MyDrive/octopus_v2/runs/full"   # 4단계에서 학습한 모델
EVAL = "/content/drive/MyDrive/octopus_v2/runs/eval_official"
TEST = "data/raw/official/official_test.jsonl"
os.makedirs(EVAL, exist_ok=True)

---
## 6-0. 엑셀 내려받기 → `official_test.jsonl`
`[우리 RULES 를 어기는 공식 정답]` 1개 (`'+12345678900,555'`, 내선번호) 는 정답을 고치지 않고 그대로 둔다.

In [ ]:
!python -m src.official_bench convert
!cp {TEST} {EVAL}/official_test.jsonl

---
## 6-1. 채점 기준 점검 (GPU 필요 없음)
엑셀에 nexa 가 기록한 Octopus-0~3 출력을 우리 채점기로 다시 매겨 nexa 의 T/F 와 비교한다.
판정이 다른 줄이 출력된다 (nexa=T 인데 우리는 오답인 경우가 대부분: 따옴표 깨짐, `72.0`, 검색어 범위 등).

In [ ]:
!python -m src.official_bench check

---
## 6-2. 우리 모델로 210개 생성
**예측**: 엄격 정확도는? 온도(`72` vs 우리 `72.0`) 10개는 엄격 채점에서 틀릴 것. negative 10개는?
> (여기에 예측)

In [ ]:
!python -m src.evaluate generate --model {RUN_DIR} --test-file {TEST} --out {EVAL}/ours/preds.jsonl

In [ ]:
!python -m src.evaluate score --preds {EVAL}/ours/preds.jsonl --test-file {TEST}

---
## 6-3. 공개 모델 `NexaAIDev/Octopus-v2` 로 210개 생성
엑셀의 Octopus-0 (1K per API, full) 기록과 비슷하게 나오면 우리 생성 코드가 nexa 환경을 재현한다는 확인도 된다.
(공개 모델이 Octopus-0~3 중 어느 것인지는 모델 카드에 없다 → 추정으로만 비교)

In [ ]:
!python -m src.evaluate generate --model NexaAIDev/Octopus-v2 --test-file {TEST} --out {EVAL}/public/preds.jsonl

In [ ]:
!python -m src.evaluate score --preds {EVAL}/public/preds.jsonl --test-file {TEST}

---
## 6-4. 나란히 보기
- `ours`, `public`: 이 노트북에서 생성 → 우리 채점기
- `nexa_O0` ~ `nexa_O3`: 엑셀에 기록된 Octopus-0~3 출력 → 우리 채점기 (지연 시간 없음)
- 맨 아래 `nexa 판정` 은 엑셀의 T/F 비율 (= 논문 표 1 숫자)

In [ ]:
import json
from src.evaluate import METRIC_NAMES, judge, summarize
from src.official_bench import OCTOPUS_COLUMNS, load_sheet
from src.preprocess.prompt import load_jsonl
from pathlib import Path

test = load_jsonl(Path(TEST))
sheet = load_sheet()
cols = {n: json.load(open(f"{EVAL}/{n}/summary.json", encoding="utf-8"))["전체"] for n in ("ours", "public")}
nexa_tf = {}
for i, name in enumerate(OCTOPUS_COLUMNS.values()):
    scored = [{**row, **judge(row, s[name]["output"] or "")} for s, row in zip(sheet, test)]
    cols[f"nexa_O{i}"] = summarize(scored)
    nexa_tf[f"nexa_O{i}"] = sum(s[name]["result"] in ("T", "t") for s in sheet) / len(sheet)

print(f"{'':<36}" + "".join(f"{n:>10}" for n in cols))
for key, label in METRIC_NAMES.items():
    print(f"  {label:<34}" + "".join(f"{cols[n][key]:>10}" if key == "n" else f"{cols[n][key]:>10.2%}" for n in cols))
print(f"  {'nexa 판정 (논문 표 1)':<34}" + "".join(f"{nexa_tf[n]:>10.2%}" if n in nexa_tf else f"{'':>10}" for n in cols))

### 우리 모델과 공개 모델의 결과가 다른 질의
한쪽만 맞힌 줄. 랩탑에서 자세히 보려면 `{EVAL}/{ours,public}/errors.csv`.

In [ ]:
sc = {n: load_jsonl(Path(f"{EVAL}/{n}/scored.jsonl")) for n in ("ours", "public")}
for a, b in zip(sc["ours"], sc["public"]):
    if (a["outcome"] == "correct") != (b["outcome"] == "correct"):
        print(f"[{a['index']}] {a['query'][:90]}")
        print(f"   정답   {a['call']}")
        print(f"   ours   {a['outcome']:<26} {a['pred_call']}")
        print(f"   public {b['outcome']:<26} {b['pred_call']}")